# 06_app
Captured 2026-11-21 23:50 UTC

**The app serves all four screens as the signed-in user** (https://shortage-radar-7405619902341965.5.azure.databricksapps.com). Radar shows 2 active disruptions; Impact lists the SKUs for V100103-2026-11-20; Approvals reads the Lakebase workflow; Ask answered in 40.5 s via agent:radar-router -> supply_exposure.


In [1]:
-- code: every request runs as the signed-in user; Lakebase via app SP + app.user
# src/app/backend/main.py
def user_ctx(req: Request) -> tuple[WorkspaceClient, str, str]:
    token = req.headers.get("x-forwarded-access-token")
    email = req.headers.get("x-forwarded-email")
    if not token or not email:
        raise HTTPException(401, "Sign in through Databricks Apps")
    return WorkspaceClient(host=HOST, token=token, auth_type="pat"), email, token

# src/app/backend/main.py
def pg(email: str) -> psycopg.Connection:
    cred = _sp.postgres.generate_database_credential(endpoint=LAKEBASE_ENDPOINT)
    host = _sp.postgres.get_endpoint(name=LAKEBASE_ENDPOINT).status.hosts.host
    conn = psycopg.connect(host=host, dbname="databricks_postgres", user=_sp.config.client_id,
                           password=cred.token, sslmode="require", autocommit=True)
    conn.execute("SELECT set_config('app.user', %s, false)", (email,))

responses below

In [2]:
-- code: Ask forwards the user's token to the routing agent
# src/app/backend/main.py
if AGENT_URL:
    try:
        body = {"id": str(uuid.uuid4()), "session_id": q.session_id or str(uuid.uuid4()),
                "input": [{"role": "user", "content": q.question}]}
        r = httpx.post(f"{AGENT_URL}/api/invocations", json=body, timeout=170,
                       headers={"Authorization": f"Bearer {token}"})

Ask response at the end

In [3]:
-- Radar: who am I (entitlements)
GET /api/me

{"email": "maggie.barrett@databricks.com", "persona": "supply_exec", "region": "*", "can_see_commercial": "true", "can_approve": "true"}

In [4]:
-- Radar: KPIs from the metric view
GET /api/kpis

[{"region": "APAC", "skus": "98", "avg_days_of_cover": "130.4", "skus_under_60_days": "2", "single_source_share": "0.326"}, {"region": "EU", "skus": "90", "avg_days_of_cover": "125.0", "skus_under_60_days": "4", "single_source_share": "0.452"}, {"region": "US", "skus": "100", "avg_days_of_cover": "128.4", "skus_under_60_days": "1", "single_source_share": "0.526"}]

In [5]:
-- Radar: active disruptions
GET /api/disruptions

[{"disruption_id": "V100103-2026-11-20", "vendor_id": "V100103", "vendor_name": "Larchmont Synthesis", "event_type": "WARNING_LETTER", "event_date": "2026-11-20", "skus": "29", "single_source_skus": "25", "first_stockout": "2027-01-11", "revenue_at_risk": "3.0045987E7"}, {"disruption_id": "V100106-2026-11-18", "vendor_id": "V100106", "vendor_name": "Dunmore Fine Chemicals", "event_type": "IMPORT_ALERT", "event_date": "2026-11-18", "skus": "9", "single_source_skus": "0", "first_stockout": "2027-01-20", "revenue_at_risk": "0.0"}]

In [6]:
-- Impact: SKUs for the top disruption (first 3)
GET /api/disruptions/V100103-2026-11-20

[{"sku_id": "SKU-0118", "drug": "morphine", "region": "EU", "site_id": "EU-DE1", "api_id": "M-API-015", "single_source": "true", "weekly_units": "931.0", "cover_days_before": "73.2", "cover_days_without_vendor": "52.3", "requalification_days": "180", "projected_stockout_date": "2027-01-11", "supply_gap_days": "127.7", "revenue_at_risk": "5582913.0"}, {"sku_id": "SKU-0119", "drug": "morphine", "region": "US", "site_id": "US-NC1", "api_id": "M-API-015", "single_source": "true", "weekly_units": "326.0", "cover_days_before": "75.5", "cover_days_without_vendor": "62.1", "requalification_days": "180", "projected_stockout_date": "2027-01-21", "supply_gap_days": "117.9", "revenue_at_risk": "1938810.0"}, {"sku_id": "SKU-0083", "drug": "midazolam", "region": "APAC", "site_id": "AP-IN1", "api_id": "M-API-011", "single_source": "true", "weekly_units": "1082.0", "cover_days_before": "73.8", "cover_days_without_vendor": "64.6", "requalification_days": "180", "projected_stockout_date": "2027-01-23", 

In [7]:
-- Approvals: Lakebase workflow
GET /api/actions

[{"action_id": 7, "disruption_id": "V100103-2026-11-20", "sku_id": "SKU-0118", "region": "EU", "action_type": "REALLOCATE_INVENTORY", "description": "Evidence test: move 4 weeks of EU stock", "est_revenue_protected": null, "proposed_by": "7739855e-67af-424e-a487-b03d30433d3a", "proposed_source": "PLANNER", "on_behalf_of": null, "proposed_at": "2026-11-21T23:49:38.697299+00:00", "status": "APPROVED", "decided_by": "maggie.barrett@databricks.com", "decided_at": "2026-11-21T23:49:39.171132+00:00", "decision_rationale": "Stock exists in EU-DE1; approved for 4 weeks"}, {"action_id": 5, "disruption_id": "V100103-2026-11-20", "sku_id": "SKU-0118", "region": "EU", "action_type": "REALLOCATE_INVENTOR

In [8]:
-- Ask
POST /api/ask

{"route": "agent:radar-router -> supply_exposure", "seconds": 40.5, "answer": "**SKU-0118 (morphine, EU) runs out first.** Its projected stockout is 2027-01-11, with 52.3 days of cover.\n\n| SKU | Drug | Region | Days of cover | Projected stockout | Supply gap (days) | Revenue at risk |\n|---|---|---|---|---|---|---|\n| SKU-0118 | morphine | EU | 52.3 | 2027-01-11 | 127.7 | $5.58M |\n| SKU-0152 | esmolol | APAC | 61.9 | 2027-01-20 | 0.0 | $0 |\n| SKU-0119 | morphine | US | 62.1 | 2027-01-21 | 117.9 | $1.94M |\n| SKU-0148 | esmolol | APAC | 62.1 | 2027-01-21 | 0.0 | $0 |\n| SKU-0083 | midazolam | APAC | 64.6 | 2027-01-23 | 115.4 | $3.91M |\n\n- **Where the risk is:** The two morphine SKUs and the midazolam SKU have supply gaps of 115 to 128 days. That means they stay short for months after they run out. They also carry the revenue at risk.\n- **Esmolol:** The two esmolol SKUs run out early, but their supply gap is 0 days and their revenue at risk is $0. They look lower priorit"}